# Titanic Survival Analysis
**Goal:** Identify the key factors that determined survival, then build and compare classification models to predict whether a passenger survived.

**Dataset:** 891 Titanic passengers · Kaggle competition dataset  
**Models compared:** Logistic Regression · Random Forest  
**Evaluation:** Accuracy · Precision · Recall · F1 · ROC-AUC

---
## Part 1 — Data Loading & Cleaning

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('train.csv')
print(f"Rows: {df.shape[0]}  Columns: {df.shape[1]}")
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill missing Age with median; drop Cabin (77% missing)
df['Age'] = df['Age'].fillna(df['Age'].median())
df.drop(columns=['Cabin'], inplace=True)

print("\nCleaning done. Shape:", df.shape)
df.head()

---
## Part 2 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Survival by gender
sns.barplot(x='Sex', y='Survived', data=df, ax=axes[0], errorbar=None)
axes[0].set_title('Survival Rate by Gender')
axes[0].set_ylabel('Survival Rate')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

# Survival by passenger class
sns.barplot(x='Pclass', y='Survived', data=df, ax=axes[1], errorbar=None)
axes[1].set_title('Survival Rate by Class')
axes[1].set_ylabel('')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

# Age distribution by survival
df[df['Survived'] == 1]['Age'].hist(ax=axes[2], alpha=0.6, bins=20, label='Survived', color='#2ecc71')
df[df['Survived'] == 0]['Age'].hist(ax=axes[2], alpha=0.6, bins=20, label='Did not survive', color='#e74c3c')
axes[2].set_title('Age Distribution by Survival')
axes[2].set_xlabel('Age')
axes[2].legend()

plt.suptitle('Key Survival Drivers — Titanic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print key statistics
gender = df.groupby('Sex')['Survived'].mean()
pclass = df.groupby('Pclass')['Survived'].mean()
print(f"Women survived: {gender['female']:.0%}  |  Men survived: {gender['male']:.0%}")
print(f"1st class: {pclass[1]:.0%}  |  2nd class: {pclass[2]:.0%}  |  3rd class: {pclass[3]:.0%}")

In [ ]:
# Heatmap: survival rate by gender AND class
pivot = df.pivot_table('Survived', index='Sex', columns='Pclass', aggfunc='mean')

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(pivot, annot=True, fmt='.0%', cmap='RdYlGn', linewidths=0.5,
            cbar_kws={'format': '{:.0%}'}, ax=ax)
ax.set_title('Survival Rate by Gender × Passenger Class')
ax.set_xlabel('Passenger Class')
plt.tight_layout()
plt.show()

cross = df.pivot_table('Survived', index='Sex', columns='Pclass', aggfunc='mean')
print("Women 1st class:", f"{cross.loc['female',1]:.0%}",
      "| Men 3rd class:", f"{cross.loc['male',3]:.0%}")

---
## Part 3 — Feature Engineering
Create model-ready features from the raw columns.

In [ ]:
# Encode Sex as binary
df['Sex_enc'] = (df['Sex'] == 'female').astype(int)

# Family size and alone flag
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Embarked: fill 2 missing with most common, then one-hot encode
df['Embarked'] = df['Embarked'].fillna('S')
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked', drop_first=True)
df = pd.concat([df, embarked_dummies], axis=1)

# Square-root transform Fare to reduce skew
df['FareSqrt'] = df['Fare'] ** 0.5

print("New features created:")
print(df[['Sex_enc', 'FamilySize', 'IsAlone', 'Embarked_Q', 'Embarked_S', 'FareSqrt']].head(6).to_string())

---
## Part 4 — Train / Test Split

In [ ]:
FEATURES = ['Pclass', 'Sex_enc', 'Age', 'FamilySize', 'IsAlone',
            'FareSqrt', 'Embarked_Q', 'Embarked_S']
TARGET   = 'Survived'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale for Logistic Regression (RF doesn't need it)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Survival rate — train: {y_train.mean():.1%}  test: {y_test.mean():.1%}")

---
## Part 5 — Model Training
Two models are trained and their predictions compared on the held-out test set.

In [ ]:
# Logistic Regression (uses scaled features)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

# Random Forest (tree-based — no scaling needed)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

# Predictions
y_pred_lr = lr.predict(X_test_sc)
y_pred_rf = rf.predict(X_test)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Both models trained.")

---
## Part 6 — Evaluation

In [ ]:
def score_model(y_true, y_pred, y_prob):
    return {
        'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Recall'   : round(recall_score(y_true, y_pred), 4),
        'F1'       : round(f1_score(y_true, y_pred), 4),
        'ROC-AUC'  : round(roc_auc_score(y_true, y_prob), 4),
    }

results = pd.DataFrame({
    'Logistic Regression': score_model(y_test, y_pred_lr, y_prob_lr),
    'Random Forest'      : score_model(y_test, y_pred_rf, y_prob_rf),
}).T

print("=== Model Comparison ===")
print(results.to_string())

In [ ]:
# ROC Curve comparison
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, y_prob_lr, name='Logistic Regression', ax=ax)
RocCurveDisplay.from_predictions(y_test, y_prob_rf, name='Random Forest', ax=ax)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline')
ax.set_title('ROC Curve — Logistic Regression vs Random Forest')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = ['Did not survive', 'Survived']
for ax, y_pred, name in zip(axes,
                              [y_pred_lr, y_pred_rf],
                              ['Logistic Regression', 'Random Forest']):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=labels
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)

plt.suptitle('Confusion Matrices (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Random Forest feature importance
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
importances.plot(kind='barh', ax=ax, color='#6c63ff', edgecolor='white')
ax.set_title('Random Forest — Feature Importances')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

top = importances.idxmax()
print(f"\nStrongest predictor: {top} ({importances[top]:.1%} of model decision weight)")

---
## Summary

| | Logistic Regression | Random Forest |
|---|---|---|
| Strengths | Interpretable coefficients, fast | Handles non-linearity, robust to outliers |
| Weaknesses | Assumes linear boundaries | Less interpretable |

**Key finding:** Both models confirm that `Sex` is the dominant predictor of survival, followed by `Pclass` and `Fare`. The survival gap between women and men (74% vs 19%) was so large that even a simple logistic model captures most of the signal.

**Limitation:** The dataset is the training split from a Kaggle competition — test labels are not available. Scores reflect an 80/20 split of the provided training data only.